In [133]:
# IMPORTS
import numpy as np
import pandas as pd
import requests

#Fin Data Sources
import yfinance as yf
import pandas_datareader as pdr

#Data viz
import plotly.graph_objs as go
import plotly.express as px

import time
from datetime import date

# for graphs
import matplotlib.pyplot as plt

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

### Question 1: IPO Filings Web Scraping and Data Processing

**What's the total sum ($m) of 2023 filings that happened on Fridays?**

Re-use the [Code Snippet 1] example to get the data from web for this endpoint: https://stockanalysis.com/ipos/filings/
Convert the 'Filing Date' to datetime(), 'Shares Offered' to float64 (if '-' is encountered, populate with NaNs).
Define a new field 'Avg_price' based on the "Price Range", which equals to NaN if no price is specified, to the price (if only one number is provided), or to the average of 2 prices (if a range is given).
You may be inspired by the function `extract_numbers()` in [Code Snippet 4], or you can write your own function to "parse" a string.
Define a column "Shares_offered_value", which equals to "Shares Offered" * "Avg_price" (when both columns are defined; otherwise, it's NaN)

Find the total sum in $m (millions of USD, closest INTEGER number) for all filings during 2023, which happened on Fridays (`Date.dt.dayofweek()==4`). You should see 32 records in total, 25 of it is not null.

(additional: you can read about [S-1 IPO filing](https://www.dfinsolutions.com/knowledge-hub/thought-leadership/knowledge-resources/what-s-1-ipo-filing) to understand the context)

In [134]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
}

url = "https://stockanalysis.com/ipos/filings/"
response = requests.get(url, headers=headers)

ipo_dfs = pd.read_html(response.text)

ipos_2023 = ipo_dfs[0]

ipos_2023.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Filing Date     325 non-null    object
 1   Symbol          325 non-null    object
 2   Company Name    325 non-null    object
 3   Price Range     325 non-null    object
 4   Shares Offered  325 non-null    object
dtypes: object(5)
memory usage: 12.8+ KB


In [135]:
ipos_2023.head()

,Filing Date,Symbol,Company Name,Price Range,Shares Offered
0,"May 3, 2024",TBN,Tamboran Resources Corporation,-,-
1,"Apr 29, 2024",HWEC,"HW Electro Co., Ltd.",$3.00,3750000
2,"Apr 29, 2024",DTSQ,DT Cloud Star Acquisition Corporation,$10.00,6000000
3,"Apr 26, 2024",EURK,Eureka Acquisition Corp,$10.00,5000000
4,"Apr 26, 2024",HDL,Super Hi International Holding Ltd.,-,-


In [136]:
# convert Fillings Date to datetime
ipos_2023['Filing Date'] = pd.to_datetime(ipos_2023['Filing Date'])

# choose only fillings for 2023
ipos_2023 = ipos_2023[ipos_2023['Filing Date'].dt.year == 2023]

# replace "-" with NaN in Shares Offered and to float64
ipos_2023['Shares Offered'] = ipos_2023['Shares Offered'].replace('-', np.nan).astype(float)

# Define a new field 'Avg_price' based on the "Price Range", which equals to NaN if no price is specified, to the price
# (if only one number is provided), or to the average of 2 prices (if a range is given).
#You may be inspired by the function `extract_numbers()` in [Code Snippet 4], or you can write your own function to
# "parse" a string.

# Function to parse price range string and calculate average price
def calculate_avg_price(price_range):
    # remove $ from price
    price_range = price_range.replace('$', '')

    if price_range == '-':
        return np.nan
    elif '-' in price_range:
        prices = price_range.split('-')
        return (float(prices[0]) + float(prices[1])) / 2
    else:
        return float(price_range)

# Define a new field 'Avg_price' based on the "Price Range"
ipos_2023['Avg_price'] = ipos_2023['Price Range'].apply(calculate_avg_price)

# Define a column "Shares_offered_value", which equals to "Shares Offered" * "Avg_price" (when both columns are defined; otherwise, it's NaN)
ipos_2023['Shares_offered_value'] = ipos_2023['Shares Offered'] * ipos_2023['Avg_price']

In [137]:
ipos_2023[ipos_2023['Filing Date'].dt.dayofweek == 4].shape

(32, 7)

In [138]:
# Find the total sum in $m (millions of USD, closest INTEGER number) for all filings during 2023, which happened on Fridays (`Date.dt.dayofweek()==4`).
# You should see 32 records in total, 25 of it is not null. The total sum should be 1,000,000,000 USD.
round(ipos_2023[ipos_2023['Filing Date'].dt.dayofweek == 4]['Shares_offered_value'].sum()/1e6, 0)

286.0

In [139]:
ipos_2023.head()

,Filing Date,Symbol,Company Name,Price Range,Shares Offered,Avg_price,Shares_offered_value
50,2023-12-29,LEC,Lafayette Energy Corp,$3.50 - $4.50,1200000.0,4.0,4800000.0
51,2023-12-29,EPSM,Epsium Enterprise Limited,-,NaN,NaN,NaN
52,2023-12-28,ONDR,"Sushi Ginza Onodera, Inc.",$7.00 - $8.00,1066667.0,7.5,8000002.5
53,2023-12-27,JDZG,Jiade Limited,$4.00 - $5.00,2200000.0,4.5,9900000.0
54,2023-12-22,LZMH,LZ Technology Holdings Limited,-,NaN,NaN,NaN


### Question 2:  IPOs "Fixed days hold" strategy


**Find the optimal number of days X (between 1 and 30), where 75% quantile growth is the highest?**


Reuse [Code Snippet 1] to retrieve the list of IPOs from 2023 and 2024 (from URLs: https://stockanalysis.com/ipos/2023/ and https://stockanalysis.com/ipos/2024/).
Get all OHLCV daily prices for all stocks with an "IPO date" before March 1, 2024 ("< 2024-03-01") - 184 tickers (without 'RYZB'). Please remove 'RYZB', as it is no longer available on Yahoo Finance.

Sometimes you may need to adjust the symbol name (e.g., 'IBAC' on stockanalysis.com -> 'IBACU' on Yahoo Finance) to locate OHLCV prices for all stocks. Also, you can see the ticker changes using this [link](https://stockanalysis.com/actions/changes/).
Some of the tickers (like 'DYCQ' and 'LEGT') were on the market less than 30 days (11 and 21 days, respectively). Let's leave them in the dataset; it just means that you couldn't hold them for more days than they were listed.

Let's assume you managed to buy a new stock (listed on IPO) on the first day at the [Adj Close] price]. Your strategy is to hold for exactly X full days (where X is between 1 and 30) and sell at the "Adj. Close" price in X days (e.g., if X=1, you sell on the next day).
Find X, when the 75% quantile growth (among 185 investments) is the highest.

HINTs:
* You can generate 30 additional columns: growth_future_1d ... growth_future_30d, join that with the table of min_dates (first day when each stock has data on Yahoo Finance), and perform vector operations on the resulting dataset.
* You can use the `DataFrame.describe()` function to get mean, min, max, 25-50-75% quantiles.


Additional:
* You can also ensure that the mean and 50th percentile (median) investment returns are negative for most X values, implying a wager for a "lucky" investor who might be in the top 25%.
* What's your recommendation: Do you suggest pursuing this strategy for an optimal X?

In [140]:
# https://stockanalysis.com/ipos/2023/
# https://stockanalysis.com/ipos/2024/

# retrieve the list of IPOs form 2023 and 2024
url_2023 = "https://stockanalysis.com/ipos/2023/"
response_2023 = requests.get(url_2023, headers=headers)

ipos_2023 = pd.read_html(response_2023.text)[0]

url_2024 = "https://stockanalysis.com/ipos/2024/"
response_2024 = requests.get(url_2024, headers=headers)

ipos_2024 = pd.read_html(response_2024.text)[0]

# convert dates to datetime objects
ipos_2023['IPO Date'] = pd.to_datetime(ipos_2023['IPO Date'])
ipos_2024['IPO Date'] = pd.to_datetime(ipos_2024['IPO Date'])

## stacked IPOs
stacked_ipos_df = pd.concat([ipos_2023, ipos_2024], ignore_index=True)

stacked_ipos_df.head()

,IPO Date,Symbol,Company Name,IPO Price,Current,Return
0,2023-12-27,IROH,Iron Horse Acquisitions Corp.,$10.00,$10.05,0.50%
1,2023-12-19,LGCB,Linkage Global Inc,$4.00,$2.91,-27.25%
2,2023-12-15,ZKH,ZKH Group Limited,$15.50,$12.95,-16.45%
3,2023-12-15,BAYA,Bayview Acquisition Corp,$10.00,$10.18,1.80%
4,2023-12-14,INHD,Inno Holdings Inc.,$4.00,$0.62,-84.45%


In [141]:
# no missing prices
stacked_ipos_df[stacked_ipos_df['IPO Price'].astype(str).str.find('-') >= 0]

,IPO Date,Symbol,Company Name,IPO Price,Current,Return


In [142]:
# no missing current prices
stacked_ipos_df[stacked_ipos_df['Current'].astype(str).str.find('-') >= 0]

,IPO Date,Symbol,Company Name,IPO Price,Current,Return


In [143]:
stacked_ipos_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   IPO Date      218 non-null    datetime64[ns]
 1   Symbol        218 non-null    object        
 2   Company Name  218 non-null    object        
 3   IPO Price     218 non-null    object        
 4   Current       218 non-null    object        
 5   Return        218 non-null    object        
dtypes: datetime64[ns](1), object(5)
memory usage: 10.3+ KB


In [144]:
# convert prices to numeric
stacked_ipos_df['Current'] = pd.to_numeric(stacked_ipos_df['Current'].str.replace('$', ''), errors='coerce')
stacked_ipos_df['IPO Price'] = pd.to_numeric(stacked_ipos_df['IPO Price'].str.replace('$', ''), errors='coerce')
stacked_ipos_df['Return'] = pd.to_numeric(stacked_ipos_df['Return'].str.replace('%', ''), errors='coerce') / 100

In [145]:
# price increase
stacked_ipos_df['Price Increase'] = stacked_ipos_df['Current'] - stacked_ipos_df['IPO Price']

stacked_ipos_df.head()

,IPO Date,Symbol,Company Name,IPO Price,Current,Return,Price Increase
0,2023-12-27,IROH,Iron Horse Acquisitions Corp.,10.0,10.05,0.0050,0.05
1,2023-12-19,LGCB,Linkage Global Inc,4.0,2.91,-0.2725,-1.09
2,2023-12-15,ZKH,ZKH Group Limited,15.5,12.95,-0.1645,-2.55
3,2023-12-15,BAYA,Bayview Acquisition Corp,10.0,10.18,0.0180,0.18
4,2023-12-14,INHD,Inno Holdings Inc.,4.0,0.62,-0.8445,-3.38


In [146]:
stacked_ipos_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   IPO Date        218 non-null    datetime64[ns]
 1   Symbol          218 non-null    object        
 2   Company Name    218 non-null    object        
 3   IPO Price       218 non-null    float64       
 4   Current         218 non-null    float64       
 5   Return          218 non-null    float64       
 6   Price Increase  218 non-null    float64       
dtypes: datetime64[ns](1), float64(4), object(2)
memory usage: 12.0+ KB


In [147]:
# checking for null values
stacked_ipos_df.isnull().sum()

IPO Date          0
Symbol            0
Company Name      0
IPO Price         0
Current           0
Return            0
Price Increase    0
dtype: int64

In [148]:
stacked_ipos_df.describe()

,IPO Price,Current,Return,Price Increase
count,218.000000,218.000000,218.000000,218.000000
mean,11.060229,11.182615,-0.208555,0.122385
std,11.245650,17.392571,0.645972,9.283959
min,2.500000,0.000000,-0.999600,-20.780000
25%,4.000000,1.245000,-0.733100,-3.682500
50%,8.000000,5.680000,-0.216600,-1.570000
75%,13.750000,10.857500,0.055450,0.582500
max,92.000000,120.520000,2.597500,56.800000


In [149]:
# use only the "IPO date" before March 1, 2024 ("< 2024-03-01")
stacked_ipos_df = stacked_ipos_df[stacked_ipos_df['IPO Date'] < '2024-03-01']

In [150]:
# Truncate to the first day in the month - for Bar names
stacked_ipos_df['Date_monthly'] = stacked_ipos_df['IPO Date'].dt.to_period('M').dt.to_timestamp()

# Count the number of deals for each month and year
monthly_deals = stacked_ipos_df['Date_monthly'].value_counts().reset_index().sort_values(by='Date_monthly')
monthly_deals.columns = ['Date_monthly', 'Number of Deals']

# Plotting the bar chart using Plotly Express
fig = px.bar(monthly_deals,
             x='Date_monthly',
             y='Number of Deals',
             labels={'Month_Year': 'Month and Year', 'Number of Deals': 'Number of Deals'},
             title='Number of IPO Deals per Month and Year',
             text='Number of Deals'
             )
fig.update_traces(textposition='outside', # Position the text outside the bars
                  textfont=dict(color='black',size=14), # Adjust the font size of the text
                  )
fig.update_layout(title_x=0.5) # Center the title

fig.show()

In [151]:
#get the list of all IPOs
stacked_ipos_df['IPO Date'].dt.year.value_counts()

2023    154
2024     31
Name: IPO Date, dtype: int64

In [ ]:
stacked_ipos_df['IPO Date']

In [109]:
# retrieve the list of IPOs form 2023
url_2023_changes = "https://stockanalysis.com/actions/changes/2023/"
response_changes_2023 = requests.get(url_2023_changes, headers=headers)
ipos_name_changes_2023_dfs = pd.read_html(response_changes_2023.text)[0]

# retrieve the list of IPOs form 2024
url_2024_changes = "https://stockanalysis.com/actions/changes/2024/"
response_changes_2024 = requests.get(url_2024_changes, headers=headers)
ipos_name_changes_2024_dfs = pd.read_html(response_changes_2024.text)[0]

In [129]:
# Get all OHLCV daily prices for all stocks with an "IPO date" before March 1, 2024 ("< 2024-03-01") - 184 tickers (without 'RYZB').
# Please remove 'RYZB', as it is no longer available on Yahoo Finance.

reddit = yf.download(tickers = "RDDT",
                     period = "max",
                     interval = "1d")

reddit.tail()

[*********************100%%**********************]  1 of 1 completed


,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2024-05-01,44.869999,47.700001,44.599998,45.880001,45.880001,1598100
2024-05-02,46.610001,49.639000,45.720001,47.700001,47.700001,2365000
2024-05-03,48.439999,48.700001,46.240002,46.639999,46.639999,1188300
2024-05-06,46.990002,49.869999,46.750000,48.270000,48.270000,2116700
2024-05-07,47.730000,50.330002,47.400002,49.400002,49.400002,5793800


In [131]:
# Get all OHLCV daily prices for all stocks with an "IPO date" before March 1, 2024 ("< 2024-03-01") - 184 tickers
# (without 'RYZB'). Please remove 'RYZB', as it is no longer available on Yahoo Finance.
reddit = reddit.copy().reset_index()

(32, 8)

In [110]:
ipos_name_changes_2023_dfs.head(1)

,Date,Old,New,New Company Name
0,"Dec 29, 2023",IOAC,ZCAR,Zoomcar Holdings Inc


In [111]:
ipos_name_changes_2024_dfs.head(1)

,Date,Old,New,New Company Name
0,"May 3, 2024",CONX,CNXX,Conx Corp
